In [ ]:
import traceback, json as _json
_result = {"status": "ok", "log": []}
def _log(*a):
    _result["log"].append(" ".join(str(x) for x in a))
try:
    yoy_rows = spark.sql("""
        SELECT YEAR(s.SalesDate) AS SalesYear, SUM(s.Revenue) AS Rev
        FROM fact_daily_sales s JOIN dim_product p ON s.ProductId = p.ProductId
        WHERE p.Category = 'Electronics' AND MONTH(s.SalesDate) = 9
          AND YEAR(s.SalesDate) IN (2025, 2026)
        GROUP BY YEAR(s.SalesDate)
    """).collect()
    yoy = {r["SalesYear"]: float(r["Rev"]) for r in yoy_rows}
    _log("YoY:", yoy)
    assert yoy[2025] == 981000.0, yoy
    assert yoy[2026] == 585000.0, yoy
    pct = 100.0 * (yoy[2026] - yoy[2025]) / yoy[2025]
    _log("decline pct:", round(pct, 2))
    assert -41 < pct < -40, pct

    rows = spark.sql("""
        SELECT s.StoreId, YEAR(s.SalesDate) AS SalesYear, SUM(s.Revenue) AS Rev
        FROM fact_daily_sales s JOIN dim_product p ON s.ProductId = p.ProductId
        WHERE p.Category = 'Electronics' AND MONTH(s.SalesDate) = 9
        GROUP BY s.StoreId, YEAR(s.SalesDate)
    """).collect()
    rev = {}
    for r in rows:
        rev.setdefault(r["StoreId"], {})[r["SalesYear"]] = float(r["Rev"])
    assert len(rev) == 10, len(rev)
    declines = {s: round(100.0 * (v[2026] - v[2025]) / v[2025], 1) for s, v in rev.items()}
    _log("declines:", dict(sorted(declines.items())))
    assert declines["S02"] == -54.5, declines
    assert all(d <= -30.0 for d in declines.values()), declines

    low = sorted(r["StoreId"] for r in
                 spark.sql("SELECT DISTINCT StoreId FROM fact_inventory WHERE IsLowStock = 1").collect())
    _log("low stock:", low)
    assert low == ["S01", "S02", "S03", "S04"], low

    cnt = spark.sql("SELECT COUNT(*) AS c FROM new_evidence").collect()[0]["c"]
    _log("new_evidence rows:", cnt)
    assert cnt == 0, cnt

    spend_rows = spark.sql("""
        SELECT StoreId, SUM(CampaignSpend) AS Spend FROM fact_promotion
        WHERE Month = '2026-09' AND Category = 'Electronics' GROUP BY StoreId
    """).collect()
    spend = {r["StoreId"]: float(r["Spend"] or 0) for r in spend_rows}
    _log("2026-09 spend:", dict(sorted(spend.items())))
    kept = sorted(s for s, v in spend.items() if v > 0)
    assert kept == ["S01", "S02", "S03"], kept

    _log("ALL VALIDATION CHECKS PASSED")
except Exception:
    _result["status"] = "error"
    _result["log"].append(traceback.format_exc())
import notebookutils
notebookutils.fs.put("Files/validation_result.json", _json.dumps(_result, indent=1), True)
print(_result["status"])
